In [1]:
"""Main file for running annealed Langevin dynamics for new material sampling."""
from __future__ import annotations
from pathlib import Path
from types import SimpleNamespace

import torch

from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model_egnn import CHGGen
from chggen.common.data_utils import get_scaler

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = CHGNetDataset(
    path='/home/xzdai/ceder_group/material_dircovery/chggen_old/data/perov_5/test_zpc.csv',
    name = 'A_good_name',
    prop_list = ['heat_all'],
)

100%|██████████| 50/50 [00:00<00:00, 320.43it/s]


In [3]:
lattice_scaler = get_scaler(dataset= dataset)

model_hparams ={'latent_dim': 64, 'hidden_dim': 128, 
                'predict_property': True, 'property_dim': 1,  
                'load_pretrain': True, 
                'fc_num_layers': 1, 
                'sigma_F_begin': 10.0, 'sigma_F_end': 0.01, 
                'sigma_L_begin': 1.0, 'sigma_L_end': 0.01, 
                'type_sigma_begin': 5.0, 'type_sigma_end': 0.01,
                'max_atoms': 20, 
                'num_noise_level': 1, 
                'lattice_scale_method': 'scale_length', 
                'cost_natom': 1.0, 'cost_coord': 10.0, 'cost_type': 1.0, 'cost_lattice': 10.0, 'cost_composition': 1.0, 'cost_edge': 10.0, 'cost_property': 1.0,
                'beta': 0.01,
                'teacher_forcing_lattice': True,
                'teacher_forcing_max_epoch': 1000,
                'decoder': 'egnn'}

chggen = CHGGen(
    hparams_dict = model_hparams, lattice_scaler = lattice_scaler, 
)

device = torch.device('cpu')
checkpoint_path = "/home/xzdai/ceder_group/material_dircovery/chggen_old/test_models/perov/trainer_perov.ckpt"
chggen = chggen.load_from_checkpoint(checkpoint_path = checkpoint_path)
chggen.lattice_scaler = lattice_scaler
chggen.to(device = device)


/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:655: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:619: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


CHGGen(
  (mse_composition): MSELoss()
  (wnd): wND()
  (encoder3d): CHGNet_encoder(
    (composition_model): AtomRef(
      (fc): Linear(in_features=94, out_features=1, bias=False)
    )
    (graph_converter): CrystalGraphConverter(algorithm='legacy', atom_graph_cutoff=5, bond_graph_cutoff=3)
    (atom_embedding): AtomEmbedding(
      (embedding): Embedding(94, 64)
    )
    (bond_basis_expansion): BondEncoder(
      (rbf_expansion_ag): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
      (rbf_expansion_bg): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
    )
    (bond_embedding): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_ag): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_bg): Linear(in_features=9, out_features=64, bias=False)
    (angle_basis_expansion): AngleEncoder(
      (fourier_expansion): Fourier()
    )
    (angle_embedding): Linear(in_features=9, out_features=64, bias=False)
    (atom_con

In [4]:
def sample_composition(composition_prob, num_atoms):
    """Sample composition such that it exactly satisfies composition_prob.
    
    Args:
        composition_prob (Tensor): The composition probability. The shape should be
            (num of structures, MAX_ATOMIC_NUM).
        num_atoms (Tensor): The number of atoms for each structure. The shape should
            be (num of structures).
    
    Returns:
        all_sampled_comp (Tensor): The sampled composition. The shape should be
            (num of atoms).
    """
    all_sampled_comp = []

    for comp_prob, num_atom in zip(list(composition_prob), list(num_atoms)):
        comp_num = torch.round(comp_prob * num_atom)
        atom_type = torch.nonzero(comp_num, as_tuple=True)[0]
        atom_num = comp_num[atom_type].long()

        sampled_comp = atom_type.repeat_interleave(atom_num, dim=0)

        # if the rounded composition gives less atoms, sample the rest
        if sampled_comp.size(0) < num_atom:
            left_atom_num = num_atom - sampled_comp.size(0)

            left_comp_prob = comp_prob - comp_num.float() / num_atom

            left_comp_prob[left_comp_prob < 0.] = 0.
            left_comp = torch.multinomial(
                left_comp_prob, num_samples=left_atom_num, replacement=True)
            # convert to atomic number
            left_comp = left_comp + 1
            sampled_comp = torch.cat([sampled_comp, left_comp], dim=0)

        sampled_comp = sampled_comp[torch.randperm(sampled_comp.size(0))]
        sampled_comp = sampled_comp[:num_atom]
        all_sampled_comp.append(sampled_comp)

    all_sampled_comp = torch.cat(all_sampled_comp, dim=0)
    assert all_sampled_comp.size(0) == num_atoms.sum()
    return all_sampled_comp


In [5]:
z = torch.randn(1, 64, requires_grad= True, device = device)
(num_atoms, 
pred_lengths_and_angles, 
pred_lengths, 
pred_angles, 
composition) = chggen.decode_stats(z)
num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles, composition[0]

/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:630: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


(tensor([5]),
 tensor([[ 4.7001e-01,  4.6271e-01,  4.6854e-01, -3.3318e-05,  3.5767e-05,
           2.8731e-04]], grad_fn=<AddmmBackward0>),
 tensor([[4.3180, 4.3155, 4.3175]]),
 tensor([[90., 90., 90.]]),
 tensor([7.6753e-07, 1.4844e-06, 2.3530e-05, 7.9639e-05, 1.9143e-05, 1.3707e-06,
         9.0572e-02, 4.1339e-01, 8.4708e-02, 8.8982e-07, 3.0252e-03, 4.4701e-03,
         2.8215e-03, 4.3986e-04, 7.6346e-07, 5.3164e-02, 2.0582e-06, 6.3109e-07,
         1.2438e-03, 9.2950e-03, 1.4337e-02, 7.7219e-03, 6.4466e-03, 5.0543e-03,
         5.2389e-03, 3.4719e-03, 2.4543e-03, 8.4540e-04, 1.8333e-03, 3.4105e-03,
         7.1345e-03, 5.8397e-03, 7.9941e-03, 1.0937e-06, 9.6777e-07, 9.8261e-07,
         1.6031e-03, 1.1739e-02, 9.4499e-03, 1.4325e-02, 8.8305e-03, 9.4916e-03,
         9.2784e-07, 5.9655e-03, 3.3845e-03, 4.5505e-03, 4.8278e-03, 6.5731e-03,
         1.2983e-02, 1.9844e-02, 1.2584e-02, 1.0226e-02, 5.1838e-07, 1.1699e-06,
         3.1605e-03, 1.4790e-02, 1.1875e-02, 9.3587e-07, 8.6987e-

In [16]:
sample_composition(composition, num_atoms)

tensor([ 9, 22,  7,  9,  7])

In [42]:
from types import SimpleNamespace

ld_kwargs = SimpleNamespace(
    n_step_each = 10,
    step_lr = 1e-6,
    min_sigma = 0,
    save_traj = False,
    disable_bar = False,
    compute_force = True,
    beta_c = 0,         # property update rate
    beta_f = 0,         # atomic force update rate                          
)
ld_kwargs.n_step_each

10

In [43]:
results = chggen.langevin_dynamics_guidance(
    z = z, 
    prop_guidance = torch.tensor(-0.05, device= device), 
    ld_kwargs= ld_kwargs
)

  0%|          | 0/50 [00:00<?, ?it/s]

sigma step: 0
step: 0
CaSbPbO2
[Structure Summary
Lattice
    abc : 4.317963123321538 4.315497398376473 4.317465782165527
 angles : 90.000002504478 90.00000250447812 89.9999974955219
 volume : 80.45234224279402
      A : 4.317963123321533 0.0 -1.887441669623513e-07
      B : 1.886363776293365e-07 4.315497398376465 -1.886363776293365e-07
      C : 0.0 0.0 4.317465782165527
    pbc : True True True
PeriodicSite: O (0.03905, 2.733, 0.3496) [0.009043, 0.6334, 0.08098]
PeriodicSite: Sb (0.2364, 2.168, 3.849) [0.05474, 0.5024, 0.8916]
PeriodicSite: Pb (1.409, 0.3675, 4.261) [0.3262, 0.08516, 0.987]
PeriodicSite: Ca (2.088, 0.6716, 2.538) [0.4836, 0.1556, 0.5879]
PeriodicSite: O (2.013, 3.744, 2.998) [0.4662, 0.8676, 0.6943]]
step: 1
CaSbPbO2
[Structure Summary
Lattice
    abc : 4.47571416169738 4.465282285724494 4.147268524121782
 angles : 90.72232195029089 92.17336549740402 91.97693069842542
 volume : 82.76756295062029
      A : 4.475130081176758 -0.017934484407305717 -0.07004547119140625
 

  2%|▏         | 1/50 [00:00<00:28,  1.72it/s]

step: 8
CaSbPbO2
[Structure Summary
Lattice
    abc : 4.9744936950565215 4.728777632047602 3.9702784638176727
 angles : 90.22421268246356 93.45497831333265 91.44592210732804
 volume : 93.19313739570904
      A : 4.943975448608398 -0.06303519010543823 -0.5465536117553711
      B : -0.022766131907701492 4.716763496398926 0.336097776889801
      C : 0.19264641404151917 -0.2964288294315338 3.954507350921631
    pbc : True True True
PeriodicSite: O (1.091, 2.711, 2.847) [0.1965, 0.621, 0.6942]
PeriodicSite: Sb (4.995, 0.8959, 3.448) [0.9732, 0.2648, 0.9839]
PeriodicSite: Pb (0.4694, 3.158, 3.459) [0.06623, 0.7222, 0.8225]
PeriodicSite: Ca (3.46, 3.115, 1.376) [0.688, 0.6937, 0.3841]
PeriodicSite: O (2.71, 1.024, 1.234) [0.5351, 0.2472, 0.3649]]
step: 9
CaSbPbO2
[Structure Summary
Lattice
    abc : 5.206374587834782 4.685588004849512 3.905910227304211
 angles : 85.78432171477453 94.39048091986415 90.94201402806706
 volume : 94.74213159302597
      A : 5.178440093994141 0.07547417283058167 -0

  4%|▍         | 2/50 [00:01<00:24,  1.99it/s]

...

 40%|████      | 20/50 [00:14<00:30,  1.02s/it]

sigma step: 20
step: 0
CaSbPbO2
[Structure Summary
Lattice
    abc : 3.994001632135924 3.728783638458182 3.0489394580227156
 angles : 38.68458167003442 98.56718954614024 105.96321401961805
 volume : 27.122909245019752
      A : 3.9829039573669434 0.2661362290382385 -0.13302861154079437
      B : -1.235304355621338 3.447662115097046 0.7010538578033447
      C : -0.5067151784896851 1.9224644899368286 2.311579942703247
    pbc : True True True
PeriodicSite: O (-0.5048, 2.917, 2.447) [0.08906, 0.2964, 0.9737]
PeriodicSite: Sb (2.414, 1.122, 0.8988) [0.672, 0.04219, 0.4147]
PeriodicSite: Pb (0.9205, 4.713, 2.621) [0.5985, 0.8055, 0.9239]
PeriodicSite: Ca (-0.3147, 2.135, 1.189) [0.09245, 0.3879, 0.402]
PeriodicSite: O (0.2614, 2.194, 1.242) [0.2373, 0.3741, 0.4374]]
step: 1
CaSbPbO2
[Structure Summary
Lattice
    abc : 3.9433970645003362 3.7159903278417747 3.0743342558159403
 angles : 38.91673988303032 98.21672535818455 105.29647831739177
 volume : 27.152622749964117
      A : 3.93258547782

 42%|████▏     | 21/50 [00:15<00:31,  1.07s/it]

sigma step: 21
step: 0
CaSbPbO2
[Structure Summary
Lattice
    abc : 3.864435927875969 3.711665246600488 3.016618531856149
 angles : 39.45370952385845 97.90755508405154 104.53049186840822
 volume : 26.50480555388146
      A : 3.854923963546753 0.2709718942642212 0.000713057117536664
      B : -1.1759871244430542 3.447288990020752 0.7139410376548767
      C : -0.5457789897918701 1.8396388292312622 2.3276257514953613
    pbc : True True True
PeriodicSite: O (-0.4318, 3.3, 2.566) [0.1557, 0.4267, 0.9714]
PeriodicSite: Sb (2.25, 1.396, 1.181) [0.6807, 0.09652, 0.4775]
PeriodicSite: Pb (1.097, 5.347, 2.89) [0.7205, 0.9948, 0.9365]
PeriodicSite: Ca (-0.003072, 2.441, 1.361) [0.2009, 0.4546, 0.4453]
PeriodicSite: O (-0.005176, 2.58, 1.529) [0.2109, 0.4558, 0.5171]]
step: 1
CaSbPbO2
[Structure Summary
Lattice
    abc : 3.8587645290180266 3.734476626344106 2.987372164584617
 angles : 39.05155750234215 97.58117111437855 104.79457012460375
 volume : 26.066684372035205
      A : 3.84932279586792 0

 44%|████▍     | 22/50 [00:17<00:31,  1.13s/it]

In [25]:
lattices = results['lattices']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

print(lattices.shape)
print(num_atoms)
print(frac_coords.shape)
print(atom_types.shape)

torch.Size([1, 3, 3])
tensor([5])
torch.Size([5, 3])
torch.Size([5])


In [28]:
from pymatgen.core import Structure, Lattice, Element

In [35]:
s = Structure(
    lattice = lattices[0].detach().numpy(), 
    species = atom_types.detach().numpy(), 
    coords = frac_coords.detach().numpy(),
    to_unit_cell = False,
    coords_are_cartesian = False,
)

In [38]:
s.to(filename = 'test.cif')

"# generated using pymatgen\ndata_BaPbO2F\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   3.53783707\n_cell_length_b   3.42190810\n_cell_length_c   3.94141020\n_cell_angle_alpha   89.86521189\n_cell_angle_beta   98.67320852\n_cell_angle_gamma   87.74474157\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   BaPbO2F\n_chemical_formula_sum   'Ba1 Pb1 O2 F1'\n_cell_volume   47.13147335\n_cell_formula_units_Z   1\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  O  O0  1  0.47707206  0.26089239  0.72417396  1\n  F  F1  1  0.24421930  0.76086646  0.22002952  1\n  O  O2  1  0.78258717  0.25923207  0.22578231  1\n  Ba  Ba3  1  0.10901868  0.77228439  0.21699725  1\n  Pb  Pb4  1  0.28334343  0.75352079  0.73077738  1\n"